# NB03b — Official Statistics and API Practice for Finance

**Role:** Core  
**Manual section:** T1.2 — Application of AI across financial sectors; 2.2.2.1 — Financial data sources  
**Prerequisites:** NB02

---

## Purpose of this notebook

External macro-economic data enriches financial analysis by providing context that internal datasets cannot supply. This notebook teaches you to retrieve structured data from two major public sources — **Eurostat** (European statistics) and the **World Bank** — using both raw API calls and dedicated Python libraries.

**Dataset:** Live API calls to Eurostat and World Bank, with local fallback CSV for classroom reliability.

### Learning objectives

By the end of this notebook you will be able to:

1. Call a public statistics API (raw HTTP) and parse the JSON response.
2. Use a **dedicated Python library** (`eurostat`, `world_bank_data`) to simplify data retrieval.
3. Transform nested API output into a tidy, analysis-ready DataFrame.
4. Understand why external context data matters in financial analytics.

### Learning pattern

> **Diagnose → Transform → Validate → Justify**  
> For each data source, you will: inspect the raw response (diagnose), reshape it into a clean DataFrame (transform), check correctness (validate), and explain why this data matters for finance (justify).

---

**Notebook chain:** NB01 → NB02 → NB03 → **NB03b** → NB04

In [ ]:
from urllib.request import urlopen
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

---

## Section 1 — Eurostat REST API: GDP at market prices

### Why GDP data matters in finance

GDP is the most-cited macro-economic aggregate. Finance teams use it for:

- **Portfolio context:** asset allocation shifts with regional growth trends.
- **Credit risk:** sovereign and corporate default probabilities correlate with GDP growth.
- **Regulatory reporting:** stress-test scenarios are defined in GDP shocks.

We begin with Eurostat's **REST API** — the lowest-level access method. This teaches you how external data is structured *before* convenience libraries abstract it away.

### How the Eurostat REST API works

The URL follows a structured pattern:

```
{host}/statistics/1.0/data/{dataset_code}?{filters}
```

We will query dataset `nama_10_gdp` — GDP and main components, current prices, million EUR — for six Euro-area economies from 2015 to 2023.

In [ ]:
# ── Eurostat REST API: GDP at market prices ─────────
try:
    url_eurostat = (
        "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
        "nama_10_gdp?format=JSON&freq=A&unit=CP_MEUR&na_item=B1GQ"
        "&geo=DE&geo=FR&geo=ES&geo=IT&geo=NL&geo=PT"
        "&time=2015&time=2016&time=2017&time=2018&time=2019"
        "&time=2020&time=2021&time=2022&time=2023"
    )
    result = json.load(urlopen(url_eurostat))
    print("Eurostat REST API call succeeded — live data loaded.")

    # --- Parse the JSON cube response ---
    values = result["value"]
    geo_idx   = result["dimension"]["geo"]["category"]["index"]
    time_idx  = result["dimension"]["time"]["category"]["index"]
    geo_labels = result["dimension"]["geo"]["category"]["label"]
    n_times = len(time_idx)

    records = []
    for geo_code, geo_pos in geo_idx.items():
        for time_label, time_pos in time_idx.items():
            flat_key = str(geo_pos * n_times + time_pos)
            val = values.get(flat_key, None)
            records.append({
                "country": geo_labels.get(geo_code, geo_code),
                "country_code": geo_code,
                "year": int(time_label),
                "gdp_meur": val,
            })

    gdp_df = pd.DataFrame(records).sort_values(["country_code", "year"])
    gdp_df = gdp_df.reset_index(drop=True)

except Exception as e:
    print(f"Eurostat REST API unavailable ({e}). Loading cached fallback.")
    gdp_df = pd.read_csv(DATA_DIR / "eurostat_gdp_fallback.csv")

gdp_df.head(9)

### Validate the result

Always check: did the request return data? Are values sensible? Any missing?

In [ ]:
print(f"Rows:      {len(gdp_df)}")
print(f"Countries: {gdp_df['country_code'].nunique()}")
print(f"Years:     {gdp_df['year'].min()} – {gdp_df['year'].max()}")
print(f"\nMissing values:")
print(gdp_df.isna().sum())

In [ ]:
# ── GDP evolution by country ─────────
fig, ax = plt.subplots(figsize=(9, 5))
for code, grp in gdp_df.groupby("country_code"):
    ax.plot(grp["year"].values, grp["gdp_meur"].values, marker="o", label=code)
ax.set_xlabel("Year")
ax.set_ylabel("GDP (million EUR, current prices)")
ax.set_title("GDP at market prices — selected Euro-area economies (Eurostat)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### What you just practised

- **Diagnose:** inspected Eurostat's positional JSON cube format.
- **Transform:** mapped flat indices to country × year records.
- **Validate:** checked row count, year range, and missing values.
- **Justify:** GDP data contextualises financial models with macro-economic trends.

> **Key insight:** raw APIs return data in formats designed for machines, not analysts. The transformation step — however simple — is always necessary.

---

## Section 2 — Eurostat Python library: unemployment rates

### From raw API to library abstraction

Section 1 showed how to call a REST API manually. In practice, dedicated Python libraries handle authentication, pagination, and parsing for you. The `eurostat` library wraps Eurostat's data API in a single function call.

### Why unemployment data matters in finance

- **Consumer credit:** unemployment drives default rates on retail lending.
- **Insurance:** claims patterns shift with the labour market.
- **Macro overlays:** unemployment is a standard stress-test variable in banking risk models.

In [ ]:
import eurostat

# ── Download monthly unemployment rate (seasonally adjusted) ─────
# Dataset: une_rt_m — Unemployment rate by sex and age, monthly
try:
    unemp_raw = eurostat.get_data_df("une_rt_m", flags=False)
    print(f"Eurostat library download succeeded — {unemp_raw.shape[0]:,} rows.")

    # ── Filter: total, seasonally adjusted, selected countries ────
    geo_col = [c for c in unemp_raw.columns if c.startswith("geo")][0]
    countries = ["DE", "FR", "ES", "IT", "PT", "NL"]

    unemp = unemp_raw[
        (unemp_raw["age"] == "TOTAL") &
        (unemp_raw["sex"] == "T") &
        (unemp_raw["s_adj"] == "SA") &
        (unemp_raw["unit"] == "PC_ACT") &
        (unemp_raw[geo_col].isin(countries))
    ].copy()

    # ── Melt from wide to long (one column per month → one row per obs)
    time_cols = [c for c in unemp.columns if str(c).startswith("20") or str(c).startswith("19")]
    unemp_long = unemp.melt(
        id_vars=[geo_col], value_vars=time_cols,
        var_name="period", value_name="unemployment_rate"
    )
    unemp_long = unemp_long.rename(columns={geo_col: "country_code"})
    unemp_long = unemp_long.dropna(subset=["unemployment_rate"])
    unemp_long = unemp_long.sort_values(["country_code", "period"]).reset_index(drop=True)

except Exception as e:
    print(f"Eurostat library unavailable ({e}). Loading cached fallback.")
    unemp_long = pd.read_csv(DATA_DIR / "eurostat_unemployment_fallback.csv")

unemp_long.head()

In [ ]:
print(f"Rows:      {len(unemp_long):,}")
print(f"Countries: {sorted(unemp_long['country_code'].unique())}")
print(f"Periods:   {unemp_long['period'].min()} to {unemp_long['period'].max()}")
print(f"\nMissing values:")
print(unemp_long.isna().sum())

In [ ]:
# ── Unemployment rate evolution ───────
fig, ax = plt.subplots(figsize=(10, 5))
for code, grp in unemp_long.groupby("country_code"):
    # Sample to annual (January) for cleaner visualisation
    annual = grp[grp["period"].astype(str).str.endswith("-01")].copy()
    if len(annual) == 0:
        annual = grp[grp["period"].astype(str).str.contains("M01")].copy()
    if len(annual) == 0:
        annual = grp.iloc[::12]  # fallback: every 12th row
    ax.plot(range(len(annual)), annual["unemployment_rate"].values, marker="o", label=code)
    ax.set_xticks(range(len(annual)))
    ax.set_xticklabels(annual["period"].values, rotation=45, ha="right", fontsize=7)

ax.set_xlabel("Period")
ax.set_ylabel("Unemployment rate (%)")
ax.set_title("Monthly unemployment rate (SA) — selected Euro-area economies (Eurostat)")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Comparing raw API vs library

| Aspect | Raw REST API (Section 1) | Python library (Section 2) |
|--------|-------------------------|---------------------------|
| **Code volume** | ~20 lines for parsing alone | ~5 lines to get a DataFrame |
| **Flexibility** | Full control over query parameters | Limited to library's filter options |
| **Understanding** | Forces you to learn the data structure | Abstracts the structure away |
| **Best for** | Custom queries, learning how APIs work | Rapid prototyping, repeated access |

> **Recommendation:** use the library for production work, but understand the raw API so you can debug when the library fails.

---

## Section 3 — World Bank Data: financial development indicators

### A global perspective on financial markets

The World Bank's data catalogue covers 21 thematic topics. Topic 4 — **Financial Sector** — contains indicators that are directly useful for cross-country financial analysis:

- Market capitalisation as % of GDP (stock market depth).
- Domestic credit to private sector (banking penetration).
- Interest rate spreads (banking efficiency).

We use the `world_bank_data` Python library, which wraps the World Bank API and returns pandas Series/DataFrames directly.

In [ ]:
import world_bank_data as wb

# ── Explore available topics ─────
topics = wb.get_topics()["value"]
print("World Bank data topics (selection):")
for idx in [3, 4, 7, 10, 14, 21]:
    if idx in topics.index:
        print(f"  Topic {idx:2d}: {topics[idx]}")

In [ ]:
# ── Financial Sector indicators (Topic 4) ─────
fin_indicators = wb.get_indicators(topic=4)["name"]
print(f"Financial Sector — {len(fin_indicators)} indicators available:\n")
for code, name in fin_indicators.items():
    print(f"  {code:30s}  {name}")

In [ ]:
# ── Download: Market capitalisation of listed domestic companies (% GDP)
# Indicator: CM.MKT.LCAP.GD.ZS
try:
    countries = ["ES", "FR", "DE", "US", "GB", "JP"]
    mktcap_data = {}
    for c in countries:
        series = wb.get_series("CM.MKT.LCAP.GD.ZS", country=c)
        # Extract year from multi-index (country, series, year)
        vals = {}
        for idx, val in series.items():
            year = idx[-1] if isinstance(idx, tuple) else idx
            vals[str(year)] = val
        mktcap_data[c] = vals

    mktcap_df = pd.DataFrame(mktcap_data).T
    mktcap_df.index.name = "country"

    # Reshape to long format
    mktcap_long = mktcap_df.reset_index().melt(
        id_vars="country", var_name="year", value_name="mkt_cap_pct_gdp"
    )
    mktcap_long["year"] = mktcap_long["year"].astype(int)
    mktcap_long = mktcap_long.dropna(subset=["mkt_cap_pct_gdp"])
    mktcap_long = mktcap_long.sort_values(["country", "year"]).reset_index(drop=True)
    print(f"World Bank API call succeeded — {len(mktcap_long)} observations.")

except Exception as e:
    print(f"World Bank API unavailable ({e}). Loading cached fallback.")
    mktcap_long = pd.read_csv(DATA_DIR / "worldbank_mktcap_fallback.csv")

mktcap_long.head(8)

In [ ]:
print(f"Rows:      {len(mktcap_long)}")
print(f"Countries: {sorted(mktcap_long['country'].unique())}")
print(f"Years:     {mktcap_long['year'].min()} – {mktcap_long['year'].max()}")
print(f"\nMissing values:")
print(mktcap_long.isna().sum())

In [ ]:
# ── Market capitalisation evolution ───────
fig, ax = plt.subplots(figsize=(10, 5))
for country, grp in mktcap_long.groupby("country"):
    ax.plot(grp["year"].values, grp["mkt_cap_pct_gdp"].values, marker="o", label=country)
ax.set_xlabel("Year")
ax.set_ylabel("Market capitalisation (% of GDP)")
ax.set_title("Stock market depth — Market capitalisation of listed companies (World Bank)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### What the data tells us

Market capitalisation as a percentage of GDP measures **stock market depth** — a key indicator of financial development:

- The **2008 financial crisis** shows as a sharp drop across all countries.
- The **US** consistently has the deepest stock market (often > 100% of GDP).
- **Spain** and **Germany** have smaller stock markets relative to GDP, reflecting their bank-oriented financial systems.

This type of cross-country comparison is exactly what portfolio managers and macro-economists need when building global allocation models.

---

## Section 4 — Combining sources: building a macro-financial panel

In practice, financial analysts rarely use a single data source. The real value comes from **combining** multiple sources into a coherent analytical table.

Let us merge GDP (Eurostat) and market capitalisation (World Bank) for the countries that overlap.

In [ ]:
# ── Merge GDP and market capitalisation ─────
# World Bank and Eurostat both use ISO 2-letter country codes
# Only keep countries present in both datasets

gdp_for_merge = gdp_df[["country_code", "year", "gdp_meur"]].copy()
mkt_for_merge = mktcap_long.rename(columns={"country": "country_code"})[
    ["country_code", "year", "mkt_cap_pct_gdp"]
]

macro_panel = pd.merge(
    gdp_for_merge, mkt_for_merge,
    on=["country_code", "year"],
    how="inner",
)

print(f"Merged panel: {len(macro_panel)} rows × {macro_panel.shape[1]} columns")
print(f"Countries:    {sorted(macro_panel['country_code'].unique())}")
print(f"Year range:   {macro_panel['year'].min()} – {macro_panel['year'].max()}")
print()
macro_panel.head(10)

In [ ]:
# ── Scatter: GDP vs market capitalisation ─────
fig, ax = plt.subplots(figsize=(8, 5))
for code, grp in macro_panel.groupby("country_code"):
    ax.scatter(grp["gdp_meur"].values, grp["mkt_cap_pct_gdp"].values, label=code, s=40, alpha=0.7)
ax.set_xlabel("GDP (million EUR)")
ax.set_ylabel("Market capitalisation (% of GDP)")
ax.set_title("Macro-financial panel — GDP vs stock market depth")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Why combining sources matters in finance

| Use case | How external data helps |
|----------|----------------------|
| **Portfolio commentary** | "Our fund underperformed because Euro-area GDP contracted by 6.3% in 2020" |
| **Credit risk models** | Add unemployment as an exogenous predictor of portfolio default rates |
| **Stress testing** | Define scenarios using official GDP/unemployment shocks |
| **Dashboards** | Combine internal KPIs with public macro indicators for executive reporting |

> **Key pattern:** real-world financial analysis is almost never limited to a single dataset. The ability to retrieve, clean, and merge data from multiple public APIs is a core professional skill.

---

## What you have learned

| Section | Data source | Method | Finance use |
|---------|------------|--------|-------------|
| 1 | Eurostat (GDP) | Raw REST API + JSON parsing | Macro context for portfolios and risk |
| 2 | Eurostat (Unemployment) | `eurostat` Python library | Credit risk, stress testing |
| 3 | World Bank (Market cap) | `world_bank_data` Python library | Financial development, global allocation |
| 4 | Combined panel | `pd.merge` across sources | Multi-source analytical tables |

### Three levels of API access

1. **Raw HTTP call** → full control, most code, deepest understanding.
2. **Dedicated Python library** → less code, automatic parsing, faster iteration.
3. **Combined panel** → the real deliverable; multiple sources merged into one table.

### Bridge to NB04

NB04 moves from description to analytical design: you will define a business question, choose a target variable, and build a modelling table. The external context data you learned to retrieve here can enrich that modelling table as **exogenous features** — macro-economic variables that the internal dataset cannot provide.

---

**Project chain:** NB01 → NB02 → NB03 → **NB03b** → NB04